# Clasificador de paisajes - Solución

Clasificador automático de paisajes usando una CNN con TensorFlow/Keras.

## 1. Importar librerías

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from pathlib import Path
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("Librerías importadas correctamente")

## 2. Cargar las imágenes

Recorremos las carpetas, cargamos las imágenes en memoria y las etiquetamos con los nombres de las carpetas. Redimensionamos a 32x32 para ir más rápido.

In [ ]:
# Configuración
class_names = ['mountain', 'street', 'glacier', 'buildings', 'sea', 'forest']
class_names_label = {name: i for i, name in enumerate(class_names)}
IMAGE_SIZE = (32, 32)  # Tamaño pequeño para ir más rápido

print("Clases:", class_names_label)

In [ ]:
# Definir rutas (ajusta si es necesario)
ROOT_PATH = os.getcwd()
TRAIN_PATH = Path(ROOT_PATH) / "seg_train"
TEST_PATH = Path(ROOT_PATH) / "seg_test"

print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)

In [ ]:
def read_data(path, im_size, class_names_label):
    X = []
    y = []
    for folder in os.listdir(path):
        label = class_names_label[folder]
        folder_path = os.path.join(path, folder)
        for file in os.listdir(folder_path):
            image_path = os.path.join(folder_path, file)
            image = cv2.imread(image_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, im_size)
            X.append(image)
            y.append(label)
    return np.array(X), np.array(y)

X_train, y_train = read_data(TRAIN_PATH, IMAGE_SIZE, class_names_label)
X_test, y_test = read_data(TEST_PATH, IMAGE_SIZE, class_names_label)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
X_train, y_train = shuffle(X_train, y_train, random_state=42)
print("Shuffle completado")

## 3. Investigar las imágenes

Comprobamos que hemos cargado bien los datos visualizando algunas muestras.

In [ ]:
print("Distribución de clases en train:")
print(pd.DataFrame(y_train, columns=['label'])['label'].value_counts().sort_index())

In [ ]:
# Mostrar ejemplos de cada clase
plt.figure(figsize=(12, 8))
for i, name in enumerate(class_names):
    idx = np.where(y_train == i)[0][0]
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_train[idx])
    plt.title(name)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Normalizar

In [ ]:
X_train_scal = X_train / 255.0
X_test_scal = X_test / 255.0

print("Valor máximo después de normalizar:", X_train_scal.max())
print("Shape X_train_scal:", X_train_scal.shape)

## 5. Diseñar la arquitectura de la red

Arquitectura:
- Conv2D 32 filtros 3x3 + ReLU
- MaxPooling2D 2x2
- Conv2D 64 filtros 3x3 + ReLU
- MaxPooling2D 2x2
- Flatten
- Dense 512 neuronas + ReLU
- Dense 6 (salida) + Softmax

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(6, activation='softmax')
])

model.summary()

## 6. Compilar el modelo

Optimizador Adam, clasificación multiclase con sparse_categorical_crossentropy.

In [ ]:
model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Modelo compilado correctamente")

## 7. Entrenar el modelo

Reservamos un 20% de los datos de entrenamiento para validación. 30 épocas, batch size 256.

In [ ]:
history = model.fit(
    X_train_scal, y_train,
    epochs=30,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

## 8. Representar el objeto history

In [ ]:
df_hist = pd.DataFrame(history.history)
df_hist.head()

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(df_hist['accuracy'], label='accuracy_train')
plt.plot(df_hist['val_accuracy'], label='accuracy_val')
plt.title('Accuracy vs epochs')
plt.ylabel('accuracy')
plt.xlabel('epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(df_hist['loss'], label='loss_train')
plt.plot(df_hist['val_loss'], label='loss_val')
plt.title('Loss vs epochs')
plt.ylabel('loss')
plt.xlabel('epochs')
plt.legend()

plt.tight_layout()
plt.show()

## 9. Evaluar el modelo con los datos de test

In [ ]:
test_loss, test_acc = model.evaluate(X_test_scal, y_test, verbose=1)
print(f"\nPrecisión en test: {test_acc:.4f}")
print(f"Loss en test: {test_loss:.4f}")

In [ ]:
# Obtener predicciones
predictions = model.predict(X_test_scal, verbose=1)
pred_labels = np.argmax(predictions, axis=1)
print("Predicciones shape:", predictions.shape)
print("Etiquetas predichas:", pred_labels[:10])

## 10. Representar algunos paisajes donde el modelo comete errores

In [ ]:
# Encontrar índices donde el modelo falla
errors = np.where(pred_labels != y_test)[0]
print(f"Total de errores en test: {len(errors)} de {len(y_test)} ({(len(errors)/len(y_test))*100:.2f}%)")

# Mostrar algunos errores
n_errors = min(6, len(errors))
plt.figure(figsize=(12, 8))
for i, idx in enumerate(errors[:n_errors]):
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_test[idx])
    plt.title(f"Real: {class_names[y_test[idx]]}\nPred: {class_names[pred_labels[idx]]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## 11. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de confusión')
plt.ylabel('Etiqueta real')
plt.xlabel('Etiqueta predicha')
plt.tight_layout()
plt.show()

In [ ]:
# Reporte de clasificación detallado
print("Reporte de clasificación:")
print(classification_report(y_test, pred_labels, target_names=class_names))